In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive"


 amazon_multi_dataset			     test.txt
'Colab Notebooks'			     train_subset_200k.txt
'Screenshot 2025-11-23 at 11.19.26 AM.png'   train.txt
 sentiment_model.pkl			     xlm_roberta_sentiment


In [ ]:
DATA_PATH = "/content/drive/MyDrive/amazon_multi_dataset"


In [ ]:
!ls "$DATA_PATH"


de  en	es  fr


In [ ]:
import os
from datasets import load_dataset, concatenate_datasets

DATA_PATH = "/content/drive/MyDrive/amazon_multi_dataset"
langs = ["en", "es", "fr", "de"]

def load_split(split):
    datasets = []
    for lang in langs:
        jsonl_file = f"{DATA_PATH}/{lang}/{split}.jsonl"
        json_file = f"{DATA_PATH}/{lang}/{split}.json"

        # Detect file
        if os.path.exists(jsonl_file):
            file = jsonl_file
        elif os.path.exists(json_file):
            file = json_file
        else:
            print(f"⚠️ No {split} file for {lang}. Will generate split automatically.")
            file = None

        if file:
            print(f"📂 Loading: {file}")
            ds = load_dataset("json", data_files=file, split="train")
        else:
            # Create validation from train
            train_file_jsonl = f"{DATA_PATH}/{lang}/train.jsonl"
            train_file_json = f"{DATA_PATH}/{lang}/train.json"

            # Detect train file
            train_file = train_file_jsonl if os.path.exists(train_file_jsonl) else train_file_json
            print(f"📂 Creating {split} for {lang} from: {train_file}")

            full_train = load_dataset("json", data_files=train_file, split="train")
            full_train = full_train.train_test_split(test_size=0.1)  # 10% for val.

            if split == "validation":
                ds = full_train["test"]
            else:
                ds = full_train["train"]

        # Add language column
        ds = ds.add_column("language", [lang] * len(ds))
        datasets.append(ds)

    return concatenate_datasets(datasets)

# Load datasets with automatic fixing for missing validation
train_ds = load_split("train")
val_ds = load_split("validation")
test_ds = load_split("test")

train_ds, val_ds, test_ds


📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/en/train.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/es/train.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/fr/train.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/de/train.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/en/validation.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/es/validation.jsonl
📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/fr/validation.jsonl
⚠️ No validation file for de. Will generate split automatically.
📂 Creating validation for de from: /content/drive/MyDrive/amazon_multi_dataset/de/train.jsonl


Flattening the indices:   0%|          | 0/20000 [00:00<?, ? examples/s]

⚠️ No test file for en. Will generate split automatically.
📂 Creating test for en from: /content/drive/MyDrive/amazon_multi_dataset/en/train.jsonl


Flattening the indices:   0%|          | 0/180000 [00:00<?, ? examples/s]

📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/es/test.jsonl
⚠️ No test file for fr. Will generate split automatically.
📂 Creating test for fr from: /content/drive/MyDrive/amazon_multi_dataset/fr/train.jsonl


Flattening the indices:   0%|          | 0/180000 [00:00<?, ? examples/s]

📂 Loading: /content/drive/MyDrive/amazon_multi_dataset/de/test.jsonl


(Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language'],
     num_rows: 800000
 }),
 Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language'],
     num_rows: 35000
 }),
 Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language'],
     num_rows: 370000
 }))

In [ ]:
from transformers import XLMRobertaTokenizerFast

MODEL_NAME = "xlm-roberta-base"
tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",  # Use fixed length for training
        max_length=128
    )

train_ds_tokenized = train_ds.map(tokenize_batch, batched=True)
val_ds_tokenized = val_ds.map(tokenize_batch, batched=True)
test_ds_tokenized = test_ds.map(tokenize_batch, batched=True)

train_ds_tokenized, val_ds_tokenized, test_ds_tokenized


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/370000 [00:00<?, ? examples/s]

(Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language', 'input_ids', 'attention_mask'],
     num_rows: 800000
 }),
 Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language', 'input_ids', 'attention_mask'],
     num_rows: 35000
 }),
 Dataset({
     features: ['id', 'text', 'label', 'label_text', 'language', 'input_ids', 'attention_mask'],
     num_rows: 370000
 }))

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/xlm_roberta_sentiment",
    eval_strategy="epoch",              # updated parameter name
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)


In [ ]:
print("Train labels:", set(train_ds_tokenized["label"]))
print("Val labels:", set(val_ds_tokenized["label"]))
print("Test labels:", set(test_ds_tokenized["label"]))


Train labels: {0, 1, 2, 3, 4}
Val labels: {0, 1, 2, 3, 4}
Test labels: {0, 1, 2, 3, 4}


In [ ]:
def normalize_labels(example):
    if example["label"] <= 1:   # original 0 or 1 -> Negative
        return {"label": 0}
    elif example["label"] == 2: # original 2 -> Neutral
        return {"label": 1}
    else:                       # original 3 or 4 -> Positive
        return {"label": 2}

train_ds = train_ds.map(normalize_labels)
val_ds   = val_ds.map(normalize_labels)
test_ds  = test_ds.map(normalize_labels)


Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/370000 [00:00<?, ? examples/s]

In [ ]:
print("Train:", set(train_ds["label"]))
print("Val:", set(val_ds["label"]))
print("Test:", set(test_ds["label"]))


Train: {0, 1, 2}
Val: {0, 1, 2}
Test: {0, 1, 2}


In [ ]:
from transformers import XLMRobertaTokenizerFast

tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)


Map:   0%|          | 0/800000 [00:00<?, ? examples/s]

Map:   0%|          | 0/35000 [00:00<?, ? examples/s]

Map:   0%|          | 0/370000 [00:00<?, ? examples/s]

In [ ]:
cols = ["text", "language", "id", "label_text"]

for col in cols:
    if col in train_ds.column_names:
        train_ds = train_ds.remove_columns(col)
        val_ds = val_ds.remove_columns(col)
        test_ds = test_ds.remove_columns(col)


In [ ]:
train_small = train_ds.shuffle(seed=42).select(range(100000)) # use 100k instead of 800k
val_small   = val_ds.shuffle(seed=42).select(range(20000))    # reduce validation


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_small,
    eval_dataset=val_small,
    compute_metrics=compute_metrics,
)


In [ ]:
from transformers import pipeline

sentiment_model = pipeline("text-classification", model=model, tokenizer=tokenizer)

examples = [
    "I love this product! It works perfectly 😍",
    "This item is terrible and broke after one day 😡",
    "It’s okay, not bad but not great either."
]

for text in examples:
    print(text, "→", sentiment_model(text)[0])


Device set to use cuda:0


I love this product! It works perfectly 😍 → {'label': 'LABEL_2', 'score': 0.9770687222480774}
This item is terrible and broke after one day 😡 → {'label': 'LABEL_0', 'score': 0.9410403966903687}
It’s okay, not bad but not great either. → {'label': 'LABEL_1', 'score': 0.785728931427002}


In [ ]:
test_results = trainer.evaluate(test_ds)
test_results


{'eval_loss': 0.5715380311012268,
 'eval_model_preparation_time': 0.0041,
 'eval_accuracy': 0.7760675675675676,
 'eval_f1': 0.7658985661043581,
 'eval_runtime': 2497.5257,
 'eval_samples_per_second': 148.147,
 'eval_steps_per_second': 18.518}

In [ ]:
model.save_pretrained("/content/drive/MyDrive/sentiment_model")
tokenizer.save_pretrained("/content/drive/MyDrive/sentiment_model")


('/content/drive/MyDrive/sentiment_model/tokenizer_config.json',
 '/content/drive/MyDrive/sentiment_model/special_tokens_map.json',
 '/content/drive/MyDrive/sentiment_model/sentencepiece.bpe.model',
 '/content/drive/MyDrive/sentiment_model/added_tokens.json',
 '/content/drive/MyDrive/sentiment_model/tokenizer.json')

In [ ]:
from transformers import pipeline, XLMRobertaTokenizerFast, XLMRobertaForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/sentiment_model"

tokenizer = XLMRobertaTokenizerFast.from_pretrained(MODEL_PATH)
model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_PATH)

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)


In [ ]:
test_output = classifier("This is great!")
print(test_output)


[{'label': 'LABEL_2', 'score': 0.9879030585289001}]


In [ ]:
import gradio as gr

label_map = {
    0: "Negative 😡",
    1: "Neutral 😐",
    2: "Positive 😊"
}

def predict(text):
    try:
        res = classifier(text)[0]

        # Fix label conversion safely
        label_raw = res["label"]

        if isinstance(label_raw, str) and "LABEL" in label_raw:
            label_id = int(label_raw.replace("LABEL_", ""))
        else:
            label_id = int(label_raw)

        confidence = round(float(res["score"]), 2)

        return f"{label_map[label_id]} (Confidence: {confidence})"

    except Exception as e:
        return f"⚠️ Error: {str(e)}"


demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type your sentence here..."),
    outputs="text",
    title="🌍 Multilingual Sentiment Analyzer"
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7f90eb914af18c90f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
